In [125]:

%load_ext autoreload
%autoreload 2


from pathlib import Path
import sys

# Walk up from the current folder until the one containing the cf/ package
ROOT = Path.cwd()
while not (ROOT / "cf").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from cf.data import RatingData
from cf.graph import RatingGraphs
from cf.metrics import rmse
from cf.models.als import ALSRecommender
from cf.models.imc_gae import IMCGAE
from cf.trainer import Trainer
from experiments.paths import TEST_DATA_PATH, TRAIN_DATA_PATH

print("repo root:", ROOT)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
repo root: /Users/raphaelbreaud/Desktop/Project/GNN_paper


In [126]:
R_train = np.load(TRAIN_DATA_PATH)
R_test = np.load(TEST_DATA_PATH)

print(R_train)
print(np.shape(R_train))
print(R_test)
print(np.shape(R_test))

[[ 4. nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan  2. nan ... nan nan nan]
 [ 3. nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]
(610, 4980)
[[nan nan  4. ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan  2. ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [ 5. nan nan ... nan nan nan]]
(610, 4980)


In [127]:
data_train = RatingData(R_train)
data_test = RatingData(R_test)


print(data_train.table)
print(np.shape(data_train.table))
print(data_test.table)
print(np.shape(data_test.table))

[[ 4. nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan  2. nan ... nan nan nan]
 [ 3. nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]
(610, 4980)
[[nan nan  4. ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan  2. ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [ 5. nan nan ... nan nan nan]]
(610, 4980)


In [128]:
# K-fold cross-validation of the ALS penalties:
#   lam       penalty on the factor matrices (stage 2, ALS)
#   lam_bias  penalty of the bias model      (stage 1)
# Uses data_train only; the test set plays no part in the choice.
import itertools, time
import numpy as np

lams      = [2, 5, 10, 20, 40]
lam_biases = [2, 5, 10, 20]
n_factors, n_iters, k, seed = 20, 15, 5, 0

folds = np.array_split(np.random.default_rng(seed).permutation(len(data_train.ratings)), k)

# One training table per fold: the fold's ratings are hidden (set to NaN)
tables = []
for fold in folds:
    table = data_train.table.astype(np.float64).copy()
    table[data_train.users[fold], data_train.items[fold]] = np.nan
    tables.append(table)

scores = np.zeros((len(lams), len(lam_biases), k))
start = time.time()
for (a, lam), (b, lam_bias) in itertools.product(enumerate(lams), enumerate(lam_biases)):
    for f, fold in enumerate(folds):
        m = ALSRecommender(n_factors=n_factors, lam=lam, lam_bias=lam_bias,
                           n_iters=n_iters, random_state=seed).fit(tables[f])
        pred = m.predict(data_train.users[fold], data_train.items[fold])     # already clipped
        scores[a, b, f] = np.sqrt(np.mean((pred - data_train.ratings[fold]) ** 2))
    print(f"lam {lam:<6g} lam_bias {lam_bias:<6g} RMSE {scores[a, b].mean():.4f} "
          f"+/- {scores[a, b].std():.4f}   ({time.time() - start:.0f} s)")

# Mean RMSE as a table: one row per lam, one column per lam_bias
mean = scores.mean(axis=2)
print("\nlam \\ lam_bias" + "".join(f"{v:>9g}" for v in lam_biases))
for a, lam in enumerate(lams):
    print(f"{lam:<14g}" + "".join(f"{v:>9.4f}" for v in mean[a]))

a, b = np.unravel_index(np.argmin(mean), mean.shape)
best_lam, best_lam_bias = lams[a], lam_biases[b]
print(f"\nbest: lam = {best_lam}, lam_bias = {best_lam_bias}, "
      f"cross-validated RMSE = {mean[a, b]:.4f}")

lam 2      lam_bias 2      RMSE 0.9512 +/- 0.0082   (10 s)
lam 2      lam_bias 5      RMSE 0.9559 +/- 0.0088   (20 s)
lam 2      lam_bias 10     RMSE 0.9638 +/- 0.0089   (29 s)
lam 2      lam_bias 20     RMSE 0.9717 +/- 0.0080   (39 s)
lam 5      lam_bias 2      RMSE 0.8983 +/- 0.0082   (49 s)
lam 5      lam_bias 5      RMSE 0.8989 +/- 0.0081   (59 s)
lam 5      lam_bias 10     RMSE 0.9043 +/- 0.0078   (68 s)
lam 5      lam_bias 20     RMSE 0.9115 +/- 0.0072   (78 s)
lam 10     lam_bias 2      RMSE 0.8830 +/- 0.0076   (88 s)
lam 10     lam_bias 5      RMSE 0.8828 +/- 0.0073   (98 s)
lam 10     lam_bias 10     RMSE 0.8881 +/- 0.0072   (107 s)
lam 10     lam_bias 20     RMSE 0.8965 +/- 0.0069   (117 s)
lam 20     lam_bias 2      RMSE 0.8834 +/- 0.0073   (126 s)
lam 20     lam_bias 5      RMSE 0.8839 +/- 0.0072   (136 s)
lam 20     lam_bias 10     RMSE 0.8914 +/- 0.0074   (145 s)
lam 20     lam_bias 20     RMSE 0.9044 +/- 0.0078   (154 s)
lam 40     lam_bias 2      RMSE 0.8834 +/- 0.0073 

In [129]:
model = ALSRecommender(n_factors=n_factors, lam=best_lam, lam_bias=best_lam_bias,
                       n_iters=n_iters).fit(data_train.table)
pred = model.predict(data_test.users, data_test.items)
print(f"RMSE test : {rmse(pred, data_test.ratings):.4f}")

RMSE test : 0.8720
